# Score cited answers

Measure whether an answer's citations point at the evidence, with two scorers side by side: overlap with human-annotated evidence spans, and a model that checks whether each cited text supports the statement it is attached to. ContractNLI has the human spans, so you can see how often the model-judged score agrees with the gold one before you rely on it for a corpus with no gold.

In this notebook you:

- Ask one generator for citations in three styles (one list for the whole answer, a marker on every sentence, and verbatim quotes) and read them into one format with character offsets.
- Score citation precision and recall against ContractNLI's evidence spans.
- Score the same citations with the definitions of ALCE (Automatic LLMs' Citation Evaluation), judged by a large language model (LLM) and by MiniCheck, a 770M-parameter checker that runs on a CPU.
- Count how often each judge agrees with the gold spans, citation by citation, and check both against two controls built to score badly.

The scoring code is in `aieng.advanced_rag.evaluation.citations`, so you can run it on your own answers. Its module docstring records the prior art (ALCE, RAGAS, TRUE, AlignScore, MiniCheck, FActScore, AttributionBench, the Text Retrieval Conference (TREC) 2024 track on retrieval-augmented generation (RAG), and the providers' citation APIs) and why each part is reused or written here.

## Choose your models and the run mode

`RUN_EXPENSIVE = False` reads the reference run. `Qwen3_6-35B-A3B`, on Vector's on-prem cluster, answered every question and judged every citation, at 20 requests per minute. MiniCheck-Flan-T5-Large judged the same citations on a CPU. Each cell that reads a model step prints which model built it and what rebuilding it on yours would cost.

`RUN_EXPENSIVE = True` answers a 40-question stratified subset with your generation model in every arm and judges it with your judging model, which defaults to the shared judge `gemini-3.8-flash` for every paid family. Gold scoring and the two controls need no model, so they are free on your answers too. Your rows compare with each other. Compare them with the reference rows only as a change of model, because the generator and the judge both differ. Set `RUN_LOCAL_CHECKER = True` as well to score your answers with MiniCheck and to build the post-hoc arm.

In [1]:
import sys
from pathlib import Path


sys.path.insert(0, str(Path.cwd()))

from aieng.advanced_rag import spend
from aieng.advanced_rag.evaluation import citations
from aieng.advanced_rag.models import choose_models
from cites import report
from cites.pipeline import Pipeline, reference_models


RUN_EXPENSIVE = False
# Score answers with MiniCheck-Flan-T5-Large on the CPU: about 3 GB of weights,
# downloaded once, and about a second per statement.
RUN_LOCAL_CHECKER = False

MODELS = choose_models(show=True)
WHICH = "subset" if RUN_EXPENSIVE else "all"
pipe = Pipeline(MODELS, live=RUN_EXPENSIVE, local_checker=RUN_LOCAL_CHECKER)

family google
  generation  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  judging     gemini-3.8-flash        $0.75 in / $3.75 out per 1M tokens; introductory price
  extraction  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  embeddings  BAAI/bge-small-en-v1.5  local
key budget unknown (the proxy did not answer)


## The surprising result first

Quoting raised citation scores on answers that were wrong. Asked for verbatim quotes over the whole agreement, the generator answered 46 of the 48 questions that the agreement does not address, almost all with "No" and a real quoted sentence. The LLM judge rated those statements supported by their quotes (mean judged recall 0.77 on those answers). The same model, asked for `[n]` markers over five retrieved chunks, declined 34 of the 48. Every quote was real and every statement followed from its quote, so citation precision and recall both looked good on answers that were wrong.

A citation score measures whether the statement follows from what it cites. It does not measure whether the statement answers the question correctly. Keep questions the corpus cannot answer in your set and score abstention beside citations, or a citation style that never declines will look like the best one.

One run, one generator (`Qwen3_6-35B-A3B`), 24 agreements. A prompt that treats "nothing to quote" as an expected outcome can bring abstention back: a two-call design that lists the supporting clauses first declines 37 of the 48 (see "Bring abstention back over the whole agreement"). It also declines 19 of the 209 answerable questions.

## The question set

ContractNLI pairs 17 fixed hypotheses about non-disclosure agreements with each agreement, and annotators labelled each pair as entailed, contradicted, or not mentioned, marking the sentences that justify the first two. The reference run takes every labelled pair from the first 24 agreements of the test split and two "not mentioned" pairs per agreement as nulls. Each hypothesis becomes a question that names all three outcomes, because a pilot asked only for Yes or No answered "No" when the agreement was silent, which reads as a contradiction.

Retrieval is the dense baseline of reference implementation 1: `bge-small-en-v1.5` over 1,200-character chunks with 200 characters of overlap, scoped to the question's own agreement, top 5.

In [2]:
items = pipe.items(WHICH)
print(f"{len(items)} questions over {len({i.doc_id for i in items})} agreements")
for label in ("Entailment", "Contradiction", "NotMentioned"):
    print(f"  {label:<14}{sum(i.label == label for i in items):>4}")
spans = [len(i.gold) for i in items if not i.is_null]
print(f"gold evidence spans per labelled question: mean {sum(spans) / len(spans):.2f}, max {max(spans)}")

# An example every style answered, with both gold sentences in the retrieved chunks.
answered = {arm: {a.query_id: a for a in pipe.answers(arm, WHICH)} for arm in ("answer", "inline", "quotes")}
inline_gold = {g.query_id: g for g in pipe.gold("inline", WHICH)}
example = next(
    i
    for i in items
    if i.label == "Entailment"
    and len(i.gold) == 2
    and all(not answered[arm][i.query_id].abstained for arm in answered)
    and inline_gold[i.query_id].shown == 2
    and report.outcome(i, answered["quotes"][i.query_id]) == "yes"
)
print()
print(example.query_id, example.label)
print("Q:", example.question)
for start, end in example.gold:
    print(f"  gold [{start}:{end}]", pipe.texts()[example.doc_id][start:end].strip())

257 questions over 24 agreements
  Entailment     171
  Contradiction   38
  NotMentioned    48
gold evidence spans per labelled question: mean 1.95, max 7
cited-answers reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
              257 calls, 353,772 prompt + 7,518 completion tokens
              rebuilding with gemini-3.5-flash-lite: about 257 calls, est. $0.12 (set RUN_EXPENSIVE = True to rebuild)
cited-answers reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
              257 calls, 348,118 prompt + 9,306 completion tokens
              rebuilding with gemini-3.5-flash-lite: about 257 calls, est. $0.13 (set RUN_EXPENSIVE = True to rebuild)
cited-answers reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
              257 calls, 367,907 prompt + 36,358 completion tokens
              rebuilding with gemini-3.5-flash-lite: about 257 calls, est. $0.20 (set RUN_EXPENSIVE = True to rebuild)

1::nda-7 Entailment
Q: Doe

## Three ways to ask for citations

The generator sees the same five chunks in every retrieved arm, and only the citation instruction changes:

- `answer` is the shipped generator (`pipeline.generate`): JSON with an answer and one list of source numbers for the whole answer.
- `inline` asks for one to three sentences, each ending in `[n]` markers, as ALCE's prompts do.
- `quotes` asks for JSON statements, each with the exact words it relies on and the source they came from. This is the prompt-level form of the quoted spans Anthropic's citations feature returns.

A fourth generated arm, `fulldoc`, uses the quote prompt with the whole agreement as a single source and no retrieval, so the gold evidence is always in view.

The example below shows one question answered in each style, with each citation's character range in the agreement.

In [3]:
for arm in ("answer", "inline", "quotes"):
    answer = answered[arm][example.query_id]
    print(f"== {arm}")
    if answer.abstained:
        print("  (abstained)")
    if answer.answer:
        print("  answer:", answer.answer)
    for statement in answer.statements:
        print("  statement:", statement.text)
        for c in statement.citations:
            print(f"    [{c.source}] {c.match:<10} chars {c.start}-{c.end}: {c.text[:110].strip()!r}")

== answer
  statement: Yes
    [3] passage    chars 5000-6200: 'any Confidential Information to its directors, officers, employees, agents or representatives (collectively "'
    [4] passage    chars 4000-5200: 'be used to compromise or expose the vulnerability of the Parties Cyber systems, processes, programs data, comm'
== inline
  statement: Yes, the agreement permits the Receiving Party to share Confidential Information with its directors, officers, employees, agents, or representatives who have a need to know for the purpose of the transaction.
    [3] passage    chars 5000-6200: 'any Confidential Information to its directors, officers, employees, agents or representatives (collectively "'
    [4] passage    chars 4000-5200: 'be used to compromise or expose the vulnerability of the Parties Cyber systems, processes, programs data, comm'
  statement: This disclosure is limited to those individuals and does not extend to other third parties without express permission.
    [1] passage

## Read citations into one format

Every format becomes a `CitedAnswer`: statements, each with citations that carry the cited text and its character range in the document. A `[n]` marker or a whole-answer source list cites a chunk, so its range is the chunk's 1,200 characters. A quote is located in its source by `locate_quote`, exactly, after folding case, spacing, and typographic quotes, or approximately, and its range is the quote's own. A quote that cannot be found is kept, marked `unmatched`, and falls back to its chunk. A marker naming a source the model was not shown goes to `dropped`.

In the reference run, 98% of quotes were found in their source (287 of 292), 257 of them character for character. The width of what a citation points at differs by a factor of five between the styles, which matters when you read the gold scores next.

`from_anthropic` reads the content blocks of a Messages response with citations enabled. Its character offsets are relative to each document as sent, and the parser shifts them by the passage's offset into the full document. The demonstration below needs no API call.

In [4]:
print("quotes located in their source:", report.quote_matches(pipe, WHICH))
for arm in ("answer", "inline", "quotes"):
    answers = pipe.answers(arm, WHICH)
    dropped = sum(len(a.dropped) for a in answers)
    print(
        f"{arm:<8} median cited width {report.citation_widths(pipe, arm, WHICH):>6.0f} chars, citations naming no shown passage: {dropped}"
    )

# The parser also reads a provider's citation blocks. These are built by hand in
# the shape Anthropic's Messages API returns, citing characters 0-80 of source 1.
shown = pipe.evidence(example.query_id)
cited = shown[0].text[:80]
blocks = [
    {"type": "text", "text": "The agreement says "},
    {
        "type": "text",
        "text": "what the first source says.",
        "citations": [
            {
                "type": "char_location",
                "cited_text": cited,
                "document_index": 0,
                "start_char_index": 0,
                "end_char_index": 80,
            },
        ],
    },
]
demo = citations.from_anthropic("demo", blocks, shown).citations[0]
print(f"source 1 starts at character {shown[0].provenance.start}; the citation reads as {demo.span}")
print("same text:", demo.text == pipe.texts()[example.doc_id][demo.start : demo.end])

quotes located in their source: {'exact': 257, 'fuzzy': 5, 'normalised': 25, 'unmatched': 5}
answer   median cited width   1200 chars, citations naming no shown passage: 1
inline   median cited width   1200 chars, citations naming no shown passage: 6
quotes   median cited width    205 chars, citations naming no shown passage: 2
source 1 starts at character 7000; the citation reads as (7000, 7080)
same text: True


## Score against the gold spans

A citation hits gold when its range overlaps an annotated evidence span by at least half of the shorter of the two. Measuring against the shorter range treats both granularities alike: a chunk hits when it contains half of a gold sentence, and a quote hits when half of the quote lies inside one.

- Gold precision is the share of an answer's citations that hit gold.
- Gold recall is the share of the question's gold spans that some citation hits.
- Shown recall counts only the gold spans that were in the five chunks the model saw, which separates the citation's loss from retrieval's.

Verdicts need no judge on ContractNLI: a labelled question is correct when the answer starts with the right Yes or No, and a null is correct when the answer declines.

In [5]:
arms = report.available(pipe, WHICH)
scored = {
    arm: (report.citation_report(pipe, arm, WHICH).to_dict(), report.correctness(pipe, arm, WHICH)) for arm in arms
}
print()
print(f"{'arm':<9}{'width':>7}{'gold P':>8}{'gold R':>8}{'shown R':>9}{'verdicts':>10}{'nulls declined':>16}")
for arm, (row, right) in scored.items():
    declined = round(right["null_abstention"] * right["nulls"])
    print(
        f"{arm:<9}{report.citation_widths(pipe, arm, WHICH):>7.0f}{row['gold_precision']:>8.3f}{row['gold_recall']:>8.3f}"
        f"{row['gold_shown_recall']:>9.3f}{right['verdict_accuracy']:>10.3f}{declined:>9} of {right['nulls']}"
    )

cited-answers reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
              257 calls, 634,871 prompt + 40,643 completion tokens
              rebuilding with gemini-3.5-flash-lite: about 257 calls, est. $0.29 (set RUN_EXPENSIVE = True to rebuild)
citation-judgements reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
                    263 calls, 129,774 prompt + 9,259 completion tokens
                    rebuilding with gemini-3.8-flash: about 263 calls, est. $0.13 (set RUN_EXPENSIVE = True to rebuild)
citation-judgements reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
                    346 calls, 177,325 prompt + 12,542 completion tokens
                    rebuilding with gemini-3.8-flash: about 346 calls, est. $0.18 (set RUN_EXPENSIVE = True to rebuild)
citation-judgements reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
                    315 calls, 92,223 prompt + 10,956 completio

citation-judgements reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
                    269 calls, 129,438 prompt + 9,510 completion tokens
                    rebuilding with gemini-3.8-flash: about 269 calls, est. $0.13 (set RUN_EXPENSIVE = True to rebuild)


citation-judgements reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
                    230 calls, 67,913 prompt + 8,045 completion tokens
                    rebuilding with gemini-3.8-flash: about 230 calls, est. $0.08 (set RUN_EXPENSIVE = True to rebuild)

arm        width  gold P  gold R  shown R  verdicts  nulls declined
answer      1200   0.893   0.755    0.830     0.828       31 of 48
inline      1200   0.873   0.864    0.933     0.818       34 of 48
quotes       205   0.792   0.642    0.734     0.789       11 of 48
fulldoc      245   0.893   0.708    0.708     0.833        2 of 48


shuffled    1200   0.093   0.104    0.110     0.818       34 of 48
rank1       1200   0.468   0.419    0.448     0.818       34 of 48
posthoc      232   0.744   0.497    0.532     0.818       34 of 48


- The model's own citations carry information beyond the ranking. Inline citations hit gold 0.873 of the time. Pointing every statement at the top-ranked chunk (`rank1`) hits 0.468, and moving each citation to another shown chunk (`shuffled`) hits 0.093.
- Chunk-sized citations hit gold more often than quotes. `inline` and `quotes` share the retrieved chunks, and the quote arm's precision is lower (0.792 against 0.873) while pointing at a fifth of the text. Compare precision across citation widths only with the width in view.
- With the whole agreement in view, quotes hit gold 0.893 of the time at quote width, the best precision per character here, and verdict accuracy is the highest of the arms (0.833). It is also the arm that almost never declines on a null.
- `posthoc` throws away the generator's markers and lets MiniCheck attach the best-supported sentence after the fact. It reaches 0.744 gold precision at sentence width without the generator citing anything, below the generator's own quotes.

## Bring abstention back over the whole agreement

The `fulldoc` arm declined 2 of the 48 questions the agreement does not address. Three prompt designs test whether quoting can keep abstention. Each runs on the same 257 questions with the same generator as the reference run:

- `fulldoc-sentinel` keeps the one-call quote prompt and adds a quote that means no clause supports the statement, `NO SUPPORTING CLAUSE`. A statement that rests on it is dropped, and an answer left with no statement counts as a decline.
- `fulldoc-extract` first asks for the clauses that settle the question's proposition, and declines when the list is empty. A second call writes the answer from those clauses and quotes them.
- `fulldoc-check` keeps the reference answers and asks once more whether the quotes state the proposition or only share its topic. It declines when they only share the topic.

Three more designs went untested: requiring a quote to share content words with the hypothesis, a quote-free yes, no, or not-stated classifier ahead of the citation step, and sampling several answers and declining when they disagree.

The cell below reads the reference run's cache, not your models, so it prints the same numbers whatever `RUN_EXPENSIVE` is.

In [6]:
# Always the on-prem reference run, so a live notebook starts no jobs on paid models.
reference = Pipeline(reference_models(), live=False, show=False)
explored = report.explore_summary(reference)
print(
    f"{'arm':<18}{'nulls declined':>16}{'labelled declined':>19}{'verdict acc':>13}{'gold P':>8}{'judged R':>10}{'answered nulls that say silent':>32}"
)
for arm, row in explored["abstention"].items():
    right = row["correctness"]
    declined = round(right["null_abstention"] * right["nulls"])
    print(
        f"{arm:<18}{declined:>13} of {right['nulls']}{row['labelled_declined']:>15} of {right['labelled']}"
        f"{right['verdict_accuracy']:>13.3f}{row['llm']['gold_precision']:>8.3f}{row['llm']['judged_recall']:>10.3f}"
        f"{row['null_answered_text_says_silent']:>20} of {right['nulls'] - declined}"
    )

arm                 nulls declined  labelled declined  verdict acc  gold P  judged R  answered nulls that say silent
fulldoc                       2 of 48              1 of 209        0.833   0.893     0.881                  28 of 46
fulldoc-sentinel             20 of 48              1 of 209        0.818   0.890     0.886                  11 of 28
fulldoc-extract              37 of 48             19 of 209        0.842   0.932     0.940                   2 of 11
fulldoc-check                 5 of 48              5 of 209        0.813   0.892     0.898                  26 of 43


The two-call design moves abstention most. `fulldoc-extract` declines 37 of the 48 (against 2) and also declines 19 of the 209 answerable questions. Its quotes are more precise (gold precision 0.932 against 0.893, judged recall 0.940 against 0.881). `fulldoc-sentinel` declines 20 of 48 and 1 of 209 answerable questions, with gold precision unchanged. `fulldoc-check` declines 5 of 48, because the second read sees only the quotes, so it cannot tell a missing clause from a present one.

A crude wording match finds 28 of the 46 answered nulls in the reference run saying, in the statement itself, that the agreement has no such clause. The quote prompt asks for a direct answer that begins with Yes or No, and "No" is how the model said it. With the extract design, 2 of the 11 answered nulls say so.

These are single runs of one generator, 48 nulls (one null is 2 points), and a regular-expression count. The sentinel result applies to this wording of the prompt. For `fulldoc-check` to win, the checker would need the whole agreement beside the quotes, and the cheapest test is that second read with the agreement attached, about 250 calls. For `fulldoc-extract` to lose fewer answerable questions, its first step would need higher recall on clauses, which a retrieval-assisted first step over the top-ranked chunks would test at about 500 calls.

## Score with a support judge

Without gold spans, ALCE scores citations with a support check:

- Citation recall of a statement is 1 when it has a citation and its cited texts, joined, fully support it.
- A citation is irrelevant when it does not support the statement alone and the other citations still do. Its precision is 1 when the statement has recall and the citation is not irrelevant.

The LLM judge returns TREC RAG 2024's three levels (full, partial, none) and ALCE reads only `full` as support. MiniCheck reads the statement alone, without the question, and returns a probability, thresholded at 0.5. The table prints the gold columns again beside the judged ones.

In [7]:
print("ALCE recall and precision, judged by", MODELS.judging.model if RUN_EXPENSIVE else "the reference judge")
print(report.arm_table(pipe, WHICH, "llm"))
print()
print("the same answers judged by MiniCheck-Flan-T5-Large")
print(report.arm_table(pipe, WHICH, "minicheck"))

ALCE recall and precision, judged by the reference judge
| Arm | Answered | Citations per answer | Quotes found | Gold precision | Gold recall | Judged recall | Judged precision |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| answer | 212 | 1.066 | n/a | 0.893 | 0.755 | 0.788 | 0.875 |
| inline | 204 | 1.485 | n/a | 0.873 | 0.864 | 0.946 | 0.940 |
| quotes | 241 | 1.212 | 0.983 | 0.792 | 0.642 | 0.851 | 0.860 |
| fulldoc | 254 | 1.130 | 1.000 | 0.893 | 0.708 | 0.881 | 0.888 |
| shuffled | 204 | 1.441 | n/a | 0.093 | 0.104 | 0.086 | 0.073 |
| rank1 | 204 | 1.319 | n/a | 0.468 | 0.419 | 0.422 | 0.422 |
| posthoc | 204 | 1.127 | n/a | 0.744 | 0.497 | 0.596 | 0.706 |

the same answers judged by MiniCheck-Flan-T5-Large


| Arm | Answered | Citations per answer | Quotes found | Gold precision | Gold recall | Judged recall | Judged precision |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| answer | 212 | 1.066 | n/a | 0.893 | 0.755 | 0.099 | 0.113 |
| inline | 204 | 1.485 | n/a | 0.873 | 0.864 | 0.799 | 0.781 |
| quotes | 241 | 1.212 | 0.983 | 0.792 | 0.642 | 0.714 | 0.722 |
| fulldoc | 254 | 1.130 | 1.000 | 0.893 | 0.708 | 0.684 | 0.687 |
| shuffled | 204 | 1.441 | n/a | 0.093 | 0.104 | 0.096 | 0.084 |
| rank1 | 204 | 1.319 | n/a | 0.468 | 0.419 | 0.370 | 0.370 |
| posthoc | 204 | 1.127 | n/a | 0.744 | 0.497 | 0.642 | 0.760 |


Both judges rank the controls last: judged recall falls from 0.946 on inline citations to 0.086 when they are shuffled, matching gold. At the level of arms, the judge tells real citations from wrong ones.

The two judges disagree sharply on one arm. The shipped generator's answers score 0.788 judged recall with the LLM judge and 0.099 with MiniCheck, on the same citations. Its statements are a bare "Yes" or "No": the LLM judge reads them with the question, and MiniCheck, which sees only the statement, has nothing to check. A claim-only checker needs statements that stand on their own, which is why the inline and quote prompts ask for them. The section "Give MiniCheck the question" tests two fixes for the bare ones. MiniCheck's scores on `posthoc` are circular, because MiniCheck chose those citations.

## How often the judge agrees with gold

Each citation gets two labels: whether it hits a gold span, and whether the judge says its text alone fully supports the statement. Agreement is the share of citations where the two match. Cohen's kappa corrects that for chance, which matters here because most citations hit gold, so a judge that always says yes already agrees often. The last two columns count the disagreements by direction.

In [8]:
print(f"{'arm':<9}{'checker':<11}{'n':>5}{'agree':>8}{'kappa':>8}{'judge yes, gold no':>20}{'gold yes, judge no':>20}")
for arm in report.available(pipe, WHICH):
    for checker in report.CHECKERS:
        table = report.agreement_for(pipe, arm, WHICH, checker)
        if not table.total:
            continue
        print(
            f"{arm:<9}{checker:<11}{table.total:>5}{table.accuracy:>8.3f}{table.kappa or 0:>8.3f}"
            f"{table.judge_only:>20}{table.gold_only:>20}"
        )

arm      checker        n   agree   kappa  judge yes, gold no  gold yes, judge no
answer   llm          206   0.874   0.478                  12                  14
answer   minicheck    206   0.194   0.002                   2                 164
inline   llm          279   0.814   0.314                  33                  19
inline   minicheck    279   0.713   0.177                  30                  50
quotes   llm          252   0.774   0.312                  39                  18
quotes   minicheck    252   0.663   0.142                  37                  48
fulldoc  llm          236   0.792   0.103                  27                  22
fulldoc  minicheck    236   0.669   0.037                  23                  55
shuffled llm          273   0.883   0.383                  13                  19
shuffled minicheck    273   0.857   0.280                  18                  21
rank1    llm          247   0.858   0.716                  11                  24


rank1    minicheck    247   0.798   0.594                  14                  36
posthoc  llm          211   0.754   0.400                  26                  26
posthoc  minicheck    211   0.649   0.075                  45                  29


On the model's own citations over retrieved chunks, the LLM judge matches gold on 77% to 87% of citations, with kappa from 0.31 to 0.48: moderate agreement beyond chance. MiniCheck matches on 66% to 71%, with kappa from 0.14 to 0.18, and on the answer-level arm it agrees with gold on only 19% for the reason above. On the full-agreement arm, the LLM judge's raw agreement (79%) stays near the others while kappa falls to 0.10, because nearly every citation there hits gold. Agreement is highest on the controls, whose citations are mostly plain misses.

The LLM judge errs mostly in one direction: it accepts citations that miss gold (33 of 279 inline citations, 39 of 252 quotes) more often than it rejects ones that hit. A judge is reliable here for ranking arms, where it tracked gold in every row. A single verdict is right about four times in five, so check it against the source before acting on it.

In [9]:
for row in report.disagreements(pipe, "quotes", "llm", WHICH, limit=3):
    print(row["query_id"], row["label"])
    print("  statement:", row["statement"])
    print("  cited:    ", row["cited"][:200].strip())
    for gold in row["gold"]:
        print("  gold:     ", gold[:200].strip())
    print()

1::nda-1 Contradiction
  statement: The agreement defines Confidential Information to include specific types of data and information, but it does not state that all Confidential Information must be expressly identified by the Disclosing Party.
  cited:     g. any Copyright application code, source code, technical design (not released for open use), trade secret, scientific or technical information, invention, design, process, procedure, formula, improve
  gold:      For purposes of this Agreement, "Confidential Information" means any data or information that is proprietary to the Parties and not generally known to the public, whether in tangible or intangible for

1::nda-12 Entailment
  statement: The agreement restricts the Receiving Party from using the Confidential Information to manufacture or test any product embodying it, except for the specific purpose outlined in the agreement.
  cited:     The Receiving Party agrees not to use the Confidential Information in any way, or to man

Two kinds of disagreement recur. In some, the cited text does support the statement and the annotators marked a different clause, so gold is narrower than the agreement. In others, the statement answers the question wrongly ("the agreement does not explicitly state that...") and the cited text is consistent with it, so the judge is right about support and the answer is still wrong. A support check scores the second kind as a good citation.

## Give MiniCheck the question

MiniCheck scored the shipped generator's answers at 0.099 judged recall because a bare "Yes" has nothing to check. Three designs change what it reads as the claim:

- `question_in_claim=True` on `MiniCheckChecker` writes the claim as "Question: ... Answer: ...".
- `answer-standalone` has the generator rewrite each statement of the shipped answers into a sentence that names its subject, with the citations kept.
- Using the dataset's hypothesis as the claim works for entailed questions only, so it went untested.

The table gives judged recall and Cohen's kappa against gold for each reading. The last column is the LLM judge on the original statements.

In [10]:
checks = explored["minicheck"]


def cell(row):
    """Format judged recall and kappa for one set of verdicts."""
    return f"{row['scores']['judged_recall']:>6.3f} {row['agreement']['kappa'] or 0:>6.3f}"


print(f"{'arm':<18}{'claim only':>14}{'question+claim':>16}{'rewritten':>14}{'LLM judge':>14}   (judged recall, kappa)")
for arm, row in checks.items():
    if arm == "answer-standalone":
        continue
    rewritten = cell(checks["answer-standalone"]["claim_only"]) if arm == "answer" else " " * 13
    print(
        f"{arm:<18}{cell(row['claim_only']):>14}{cell(row['question_and_claim']):>16}{rewritten:>14}{cell(row['llm_judge']):>14}"
    )

arm                   claim only  question+claim     rewritten     LLM judge   (judged recall, kappa)
answer              0.099  0.002    0.528  0.218  0.613  0.174  0.788  0.478
inline              0.799  0.177    0.635  0.367                0.946  0.314
quotes              0.714  0.142    0.439  0.141                0.851  0.312
fulldoc             0.684  0.037    0.465  0.097                0.881  0.103


On the shipped generator's answers, either change lifts MiniCheck's judged recall from 0.099 to 0.528 (question in the claim) or 0.613 (rewritten), against 0.788 for the LLM judge, and raises kappa against gold from 0.002 to 0.218 or 0.174 (the LLM judge reaches 0.478). Per-citation agreement with gold rises from 19% to 65% and 69%.

On the arms whose statements already stand alone, the question makes MiniCheck stricter. Judged recall falls on `inline` (0.799 to 0.635), `quotes` (0.714 to 0.439), and `fulldoc` (0.684 to 0.465), as the count of gold hits it rejects grows (68 against 50 on `inline`). Kappa rises on `inline` (0.177 to 0.367, above the LLM judge's 0.314) and stays near 0.14 on `quotes`. Add the question when the answer text is a bare verdict, and leave it out when each statement names its subject.

This applies to MiniCheck-Flan-T5-Large, one prompt wording for the claim, and ContractNLI. For the question to help a checker on standalone statements, the checker would need training on question-and-answer pairs. The cheapest test is the same table with the hypothesis as the claim on the entailed questions, which needs no chat-model call.

## Score your own cited answers

The functions take any answer text and the passages it was shown:

1. Read the answer with `parse_inline` (markers), `parse_structured` (JSON statements or quotes), `from_grounded` (the shipped generator), or `from_anthropic` (provider citation blocks).
2. Where you have annotated evidence, score it with `score_against_gold`.
3. Score it with `judge_citations` and an `LLMSupportChecker` over your judging model, or `MiniCheckChecker` on the CPU.
4. Before you trust the judge on your corpus, label 50 citations by hand and compare them with `agreement`, as this notebook does with ContractNLI's spans.

`generate_cited(..., style="quotes")` asks your generator for quotes. The cell below scores a hand-written answer against gold, and with `RUN_EXPENSIVE` on it also judges it with your judging model.

In [11]:
# Parse your answer, score it against gold spans, then judge it.
shown = pipe.evidence(example.query_id)
gold_chunk = next(
    n
    for n, passage in enumerate(shown, start=1)
    if any(citations.overlaps(passage.provenance.span, span) for span in example.gold)
)
other = 1 if gold_chunk != 1 else 2
mine = citations.parse_inline(
    example.query_id, f"Yes, the agreement allows disclosure to advisers [{gold_chunk}][{other}].", shown
)
print(citations.score_against_gold(mine, example.gold, doc_id=example.doc_id, shown=shown))

if RUN_EXPENSIVE:
    from cites.pipeline import Clients

    judge = citations.LLMSupportChecker(Clients().completer(MODELS.judging.model, max_tokens=200), MODELS.judging.model)
    print(citations.judge_citations(mine, judge, question=example.question).to_dict())

GoldScore(query_id='1::nda-7', hits=(True, False), gold=2, covered=1, shown=2)


## What this run covers

One run, no repeats: one generator and judge (`Qwen3_6-35B-A3B`), one local checker, 257 questions over 24 ContractNLI agreements, with retrieval scoped to one agreement. The negatives are scoped to that setup.

- Quotes over retrieved chunks lost gold precision to chunk citations (0.792 against 0.873). For quotes to win, precision would have to be compared at equal width, or the gold spans would have to be the citable units. The cheapest test is giving the generator ContractNLI's own sentence spans as numbered sources, so every citation is gold-sized: about 260 generation calls.
- Post-hoc attribution by MiniCheck lost to the generator's own quotes (0.744 against 0.792 gold precision). It would win if the generator's own citations were worse, as they might be for a small paid model. The cheapest test is the `posthoc` arm on your own generator's inline answers, which costs no calls with `RUN_LOCAL_CHECKER` on.
- MiniCheck agreed with gold less than the LLM judge. Giving it the question, or rewriting bare statements to stand alone, closed part of the gap on the shipped generator's answers and made it stricter on statements that already stood alone. The hypothesis as the claim, on entailed questions, is untested.

In [12]:
print(spend.report())

this run, estimated   est. $0.00 over 0 models
